# 02 · Create two slices for an alignment experiment

In this tutorial, you will simulate expression for DLPFC slice **151675**, rotate the simulated slice by **35°**, and crop it to the original tissue window.
The two slices will partly overlap, and their spot IDs will tell you which spots correspond.

Use `dlpfc/151675.h5ad`, the same input as notebooks 00 and 04. See [setup](README.md#setup).
This example follows one rotation condition from Study 02.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import FEAST

TUTORIAL = Path.cwd() if Path.cwd().name == "tutorial" else Path.cwd() / "tutorial"
sys.path.insert(0, str(TUTORIAL))
from _utils import data_root, load_counts, gene_summary, gene_values, spatial_panel
DATA = data_root()

OUT = TUTORIAL / "outputs" / "02"
OUT.mkdir(parents=True, exist_ok=True)
print("FEAST", FEAST.__version__)

In [ ]:
from FEAST.alignment import rotate_spatial, apply_spatial_transform
reference = load_counts(DATA / "dlpfc/151675.h5ad", "ground_truth")

### 1. Simulate expression and rotate the slice

First, generate new expression counts with FEAST. Then rotate the spot coordinates while keeping a fixed rectangular window around the original slice.
Spots that move outside this window are removed, creating partial overlap.

We place the rotation center away from the middle of the window, using the Study 02 setting:
`window center + 0.5 × window half-width` along each axis.
`rotate_spatial` changes the coordinates and keeps the counts of each remaining spot unchanged.

In [ ]:
%%capture --no-stderr
base = FEAST.simulate(
    reference, seed=2026, parameter_mode="hungarian", spatial_mode="reference_rank",
    assignment_solver="scipy", assignment_blocks=False, verbose=False,
)

In [ ]:
xy = np.asarray(reference.obsm["spatial"])
plate = np.array([xy.min(axis=0), xy.max(axis=0)])
center = plate.mean(axis=0) + 0.5 * (plate[1] - plate[0]) / 2
moving = rotate_spatial(base, angle_degrees=35, center=center, plate_bounds=plate)
print(f"Retained {moving.n_obs:,}/{reference.n_obs:,} spots ({moving.n_obs/reference.n_obs:.1%})")

### 2. Check the known spot matches

Because we applied the rotation ourselves, we know how to undo it.
The next plot uses that known inverse to return the retained spots to their original positions.
This checks the simulated geometry; it does not test an alignment algorithm.

When evaluating a method such as PASTE2 or Spateo, let the method estimate the alignment from the two slices.
Reserve the known transform and spot matches for checking its result.

In [ ]:
truth = moving.uns["feast_alignment_transform"]
restored_xy = apply_spatial_transform(moving.obsm["spatial"], truth["inverse_matrix"])
matched_xy = reference[moving.obs_names].obsm["spatial"]
np.testing.assert_allclose(restored_xy, matched_xy, atol=1e-8)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
spatial_panel(axes[0], xy, reference.obs["ground_truth"], "Full reference", True)
spatial_panel(axes[1], moving.obsm["spatial"], moving.obs["ground_truth"], "35° rotation + crop", True)
spatial_panel(axes[2], restored_xy, moving.obs["ground_truth"], "Known inverse (retained spots)", True)
for ax in axes:
    ax.set_xlim(plate[0, 0], plate[1, 0])
    ax.set_ylim(plate[1, 1], plate[0, 1])
plt.show()

In [ ]:
# Export expression and geometry only; keep known transforms and labels out of method inputs.
for name, data in [("reference", reference), ("moving", moving)]:
    method_input = ad.AnnData(X=data.X.copy(), obs=pd.DataFrame(index=data.obs_names),
                             var=pd.DataFrame(index=data.var_names))
    method_input.obsm["spatial"] = data.obsm["spatial"].copy()
    method_input.write_h5ad(OUT / f"{name}.h5ad")
pd.DataFrame({"moving_id": moving.obs_names, "reference_id": moving.obs_names}).to_csv(
    OUT / "correspondence.csv", index=False,
)

**How to read the plots:** the middle panel shows the rotated slice after cropping.
The right panel returns the remaining spots to their original positions. Spots removed by cropping are still absent.

Evaluate an estimated alignment using the retained spot IDs. The exported slice files contain expression and coordinates;
`correspondence.csv` records the known matches for evaluation. Study 02 provides the full method-comparison workflow.

### Saved example

![DLPFC 151675: full reference, baseline rotation at 35°, and coordinates restored with the known inverse.](assets/02.png)

DLPFC 151675: full reference, baseline rotation at 35°, and coordinates restored with the known inverse.

This image comes from an earlier reproduction run. It is not a new result from this notebook. A fresh run may differ with the software version and environment.

<details>
<summary>Image sources</summary>

Rendered on 2026-09-04 from these existing files in `FEAST_reproduce`, using the plotting code above:

- `05_2d_conditional_transfer/data/local/dlpfc/151675.h5ad`
- `02_alignment/outputs/fixed_plate_rerun_20260806_v1/rotations/baseline_rotated_35.h5ad`

The source files were read without rerunning the simulations. These images do not show a new execution of the notebook.

</details>